# Predictive Performance

**EmbeddingGemma 2 version** of `main/code/03_2_predictive_performance_txt.ipynb`: the fine-tuned (DoubleMLDeep) product embeddings are replaced by [EmbeddingGemma 2](https://huggingface.co/google/embeddinggemma-2) embeddings (text only, Matryoshka 256 dimensions). Datasets are read from `gemma/data`, raw data from `main/data`, outputs are written to `gemma/outputs`.

In [1]:
import re
import datasets
import pandas as pd
import numpy as np
import statsmodels.api as sm

from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score

from utils.utils_data2 import (
    load_pred_and_emb,
    center_and_norm,
    get_pca,
    get_cluster,
    get_similarities,
    add_lags_and_scale_data,
)

In [2]:
txt_only = True
embeddings = True
dataframe_name = f"dataset_txt_only_{txt_only}_embeddings_{embeddings}"
df_full_train = pd.read_csv(f"../data/{dataframe_name}_train.zip")
df_full_val = pd.read_csv(f"../data/{dataframe_name}_val.zip")

columns_to_drop = [
    "Q_t-2",
    "P_bb_t-2",
    "REVIEW_COUNT_t-2",
    "RATING_t-2",
]
df_full_train = df_full_train.drop(columns=columns_to_drop)
df_full_val = df_full_val.drop(columns=columns_to_drop)

df_full_train = df_full_train.dropna()
df_full_val = df_full_val.dropna()

dummy_subcat_names = [
    category for category in df_full_val["subcat_aggregated"].unique()
]
all_time_steps = sorted([str(date) for date in df_full_val["date"].unique()])

print(f"Dummy subcat names: {dummy_subcat_names}")
print(f"All time steps: {all_time_steps}")
df_full_val.columns

Dummy subcat names: ['Residual', 'Cars & Race Cars', 'Trains & Trams', 'Train Sets', 'Trucks', 'Tractors', 'Race Tracks', 'Airplanes', 'Motor Vehicles', 'Vehicle Playsets', 'Skateboards']
All time steps: ['2023-02-27', '2023-03-27', '2023-04-24', '2023-05-22', '2023-06-19', '2023-07-17', '2023-08-14', '2023-09-11', '2023-10-09', '2023-11-06', '2023-12-04', '2024-01-01', '2024-01-29']


Index(['ASIN', 'date', 'index', 'Q_t', 'PRICE', 'P_bb_t', 'text', 'window',
       'REVIEW_COUNT', 'RATING',
       ...
       'Delta_Q_t', 'Delta_P_bb_t', 'pred_ml_l', 'pred_ml_m', 'pred_ml_l_diff',
       'pred_ml_m_diff', 'pred_ml_l_lag_1', 'pred_ml_m_lag_1',
       'pred_ml_l_diff_lag_1', 'pred_ml_m_diff_lag_1'],
      dtype='object', length=344)

## Variable Names

In [3]:
n_lags = 1

outcome = "Q_t"
treatment = "P_bb_t"

outcome_diff = "Delta_Q_t"
treatment_diff = "Delta_P_bb_t"

dummy_time_steps = all_time_steps[n_lags:]

all_dummy_controls = (
    dummy_subcat_names
    + dummy_time_steps
    + [
        "Lightning Deals: Upcoming Deal",
        "Buy Box: Is FBA",
    ]
)

dummy_baselines = [dummy_time_steps[0], "Residual"]
dummy_time_steps_wo_baseline = [t for t in dummy_time_steps if t not in dummy_baselines]
dummy_controls = [t for t in all_dummy_controls if t not in dummy_baselines]

cont_controls = [
    "RATING_t-1",
    "REVIEW_COUNT_t-1",
    "New Offer Count: Current",
    "Count of retrieved live offers: New, FBA",
    "Count of retrieved live offers: New, FBM",
]

add_controls_to_scale = [
    "RATING",
    "REVIEW_COUNT",
]

controls_pca = ["pca_0", "pca_1", "pca_2", "pca_3", "pca_4"]
controls_similarities = [
    "similarity_cluster_0",
    "similarity_cluster_1",
    "similarity_cluster_2",
    "similarity_cluster_3",
    "similarity_cluster_4",
]

additional_controls = cont_controls + dummy_controls

additional_controls_deep = [
    var for var in additional_controls if var not in dummy_subcat_names
]

controls_emb = [var for var in df_full_train.columns if "emb" in var]

In [4]:
# dataframe for all results
column_names = ["R2 Q Train", "R2 Q Test", "R2 P Train", "R2 P Test"]

results_df = pd.DataFrame(columns=column_names)
results_df_diff = pd.DataFrame(columns=column_names)

## Performance Deep Model with time independent controls

The deep models are the fine-tuned DoubleMLDeep models of the main analysis (`pred_ml_*` are taken unchanged from the datasets of the main analysis). They are reported as benchmark for the predictive performance of the EmbeddingGemma 2 features below.

In [5]:
df_dict = {
    "Train": df_full_train,
    "Test": df_full_val,
}

results_df_deep = pd.DataFrame(
    data=np.full((2, 4), np.nan),
    columns=column_names,
    index=["Deep Time Independent", "Deep Time Dependent"],
)

for df_name, df in df_dict.items():
    y = df[outcome].values
    d = df[treatment].values

    pred_ml_l = df["pred_ml_l"].values
    pred_ml_m = df["pred_ml_m"].values

    r2_ml_l = np.round(r2_score(y, pred_ml_l), 4)
    r2_ml_m = np.round(r2_score(d, pred_ml_m), 4)

    print(f"Evaluation for {df_name} set")
    print(f"R2 Outcome: {r2_ml_l}")
    print(f"R2 Treatment: {r2_ml_m}")

    pred_ml_l_lag1 = df["pred_ml_l_lag_1"].values
    pred_ml_m_lag1 = df["pred_ml_m_lag_1"].values

    r2_ml_l_lag1 = np.round(r2_score(y, pred_ml_l_lag1), 4)
    r2_ml_m_lag1 = np.round(r2_score(d, pred_ml_m_lag1), 4)

    print(f"Evaluation for {df_name} set")
    print(f"R2 Outcome (time dependent): {r2_ml_l_lag1}")
    print(f"R2 Treatment (time dependent): {r2_ml_m_lag1}")

    results_df_deep[f"R2 Q {df_name}"] = (r2_ml_l, r2_ml_l_lag1)
    results_df_deep[f"R2 P {df_name}"] = (r2_ml_m, r2_ml_m_lag1)

results_df_deep

Evaluation for Train set
R2 Outcome: 0.8539
R2 Treatment: 0.9539
Evaluation for Train set
R2 Outcome (time dependent): 0.8611
R2 Treatment (time dependent): 0.9589
Evaluation for Test set
R2 Outcome: 0.5031
R2 Treatment: 0.6574
Evaluation for Test set
R2 Outcome (time dependent): 0.6065
R2 Treatment (time dependent): 0.6489


,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
Deep Time Independent,0.8539,0.5031,0.9539,0.6574
Deep Time Dependent,0.8611,0.6065,0.9589,0.6489


In [6]:
df_dict = {
    "Train": df_full_train,
    "Test": df_full_val,
}

results_df_diff_deep = pd.DataFrame(
    data=np.full((2, 4), np.nan),
    columns=column_names,
    index=["Deep Time Independent", "Deep Time Dependent"],
)

for df_name, df in df_dict.items():
    y_diff = df["Delta_Q_t"].values
    d_diff = df["Delta_P_bb_t"].values

    pred_ml_l_diff = df["pred_ml_l_diff"].values
    pred_ml_m_diff = df["pred_ml_m_diff"].values

    r2_ml_l_diff = np.round(r2_score(y_diff, pred_ml_l_diff), 8)
    r2_ml_m_diff = np.round(r2_score(d_diff, pred_ml_m_diff), 8)

    print(f"Evaluation for {df_name} set")
    print(f"R2 Outcome: {r2_ml_l_diff}")
    print(f"R2 Treatment: {r2_ml_m_diff}")

    pred_ml_l_diff_lag1 = df["pred_ml_l_diff_lag_1"].values
    pred_ml_m_diff_lag1 = df["pred_ml_m_diff_lag_1"].values

    r2_ml_l_diff_lag1 = np.round(r2_score(y_diff, pred_ml_l_diff_lag1), 8)
    r2_ml_m_diff_lag1 = np.round(r2_score(d_diff, pred_ml_m_diff_lag1), 8)

    print(f"Evaluation for {df_name} set")
    print(f"R2 Outcome (time dependent): {r2_ml_l_diff_lag1}")
    print(f"R2 Treatment (time dependent): {r2_ml_m_diff_lag1}")

    results_df_diff_deep[f"R2 Q {df_name}"] = (r2_ml_l_diff, r2_ml_l_diff_lag1)
    results_df_diff_deep[f"R2 P {df_name}"] = (r2_ml_m_diff, r2_ml_m_diff_lag1)

results_df_diff_deep

Evaluation for Train set
R2 Outcome: 0.19432169
R2 Treatment: 0.01075996
Evaluation for Train set
R2 Outcome (time dependent): 0.10213353
R2 Treatment (time dependent): 0.01278636
Evaluation for Test set
R2 Outcome: 0.14476851
R2 Treatment: 0.01011942
Evaluation for Test set
R2 Outcome (time dependent): 0.09478898
R2 Treatment (time dependent): 0.0123968


,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
Deep Time Independent,0.194322,0.144769,0.010760,0.010119
Deep Time Dependent,0.102134,0.094789,0.012786,0.012397


## Feature generation

In [7]:
# train set
y_train = df_full_train[outcome].squeeze()
d_train = df_full_train[treatment].squeeze()
y_train_diff = df_full_train[outcome_diff].squeeze()
d_train_diff = df_full_train[treatment_diff].squeeze()

x_train = df_full_train[additional_controls]
x_train = sm.add_constant(x_train)

x_train_pca = df_full_train[additional_controls + controls_pca]
x_train_pca = sm.add_constant(x_train_pca)
x_train_sim = df_full_train[additional_controls + controls_similarities]
x_train_sim = sm.add_constant(x_train_sim)


# test set
y_test = df_full_val[outcome].squeeze()
d_test = df_full_val[treatment].squeeze()
y_test_diff = df_full_val[outcome_diff].squeeze()
d_test_diff = df_full_val[treatment_diff].squeeze()

x_test = df_full_val[additional_controls]
x_test = sm.add_constant(x_test)

x_test_pca = df_full_val[additional_controls + controls_pca]
x_test_pca = sm.add_constant(x_test_pca)
x_test_sim = df_full_val[additional_controls + controls_similarities]
x_test_sim = sm.add_constant(x_test_sim)

# update names for boosting
x_train.rename(columns=lambda x: re.sub("[^A-Za-z0-9_]+", "", x), inplace=True)
x_test.rename(columns=lambda x: re.sub("[^A-Za-z0-9_]+", "", x), inplace=True)

x_train_pca.rename(columns=lambda x: re.sub("[^A-Za-z0-9_]+", "", x), inplace=True)
x_test_pca.rename(columns=lambda x: re.sub("[^A-Za-z0-9_]+", "", x), inplace=True)

x_train_sim.rename(columns=lambda x: re.sub("[^A-Za-z0-9_]+", "", x), inplace=True)
x_test_sim.rename(columns=lambda x: re.sub("[^A-Za-z0-9_]+", "", x), inplace=True)

In [8]:
train_dict = {
    "y": y_train,
    "y_diff": y_train_diff,
    "d": d_train,
    "d_diff": d_train_diff,
    "x": x_train,
    "x_pca": x_train_pca,
    "x_sim": x_train_sim,
}

test_dict = {
    "y": y_test,
    "y_diff": y_test_diff,
    "d": d_test,
    "d_diff": d_test_diff,
    "x": x_test,
    "x_pca": x_test_pca,
    "x_sim": x_test_sim,
}

## Performance Level Data

In [9]:
feature_specifications = ["x", "x_pca", "x_sim"]
results_df_tab = pd.DataFrame()

for feature_specification in feature_specifications:
    print("\n ------------------------------------")
    print(f"Feature specification: {feature_specification}")

    print("\nOLS")
    outcome_reg = sm.OLS(train_dict["y"], train_dict[feature_specification]).fit()
    ols_outcome_train = r2_score(
        train_dict["y"], outcome_reg.predict(train_dict[feature_specification])
    )
    print(f"R2 Outcome (train): {ols_outcome_train}")
    ols_outcome_test = r2_score(
        test_dict["y"], outcome_reg.predict(test_dict[feature_specification])
    )
    print(f"R2 Outcome (test): {ols_outcome_test}")

    treatment_reg = sm.OLS(train_dict["d"], train_dict[feature_specification]).fit()
    ols_treatment_train = r2_score(
        train_dict["d"], treatment_reg.predict(train_dict[feature_specification])
    )
    print(f"R2 Treatment (train): {ols_treatment_train}")
    ols_treatment_test = r2_score(
        test_dict["d"], treatment_reg.predict(test_dict[feature_specification])
    )
    print(f"R2 Treatment (test): {ols_treatment_test}")

    print("\nBoosting")
    boost_q_outcome = LGBMRegressor(
        n_estimators=500, lr=0.02, random_state=42, verbose=-1
    )
    boost_q_outcome.fit(train_dict[feature_specification], train_dict["y"])
    r2_boost_q_train = boost_q_outcome.score(
        train_dict[feature_specification], train_dict["y"]
    )
    print(f"R2 Outcome (train): {r2_boost_q_train}")
    r2_boost_q_test = boost_q_outcome.score(
        test_dict[feature_specification], test_dict["y"]
    )
    print(f"R2 Outcome (test): {r2_boost_q_test}")

    boost_q_treatment = LGBMRegressor(
        n_estimators=500, lr=0.02, random_state=42, verbose=-1
    )
    boost_q_treatment.fit(train_dict[feature_specification], train_dict["d"])
    r2_boost_d_train = boost_q_treatment.score(
        train_dict[feature_specification], train_dict["d"]
    )
    print(f"R2 Treatment (train): {r2_boost_d_train}")
    r2_boost_d_test = boost_q_treatment.score(
        test_dict[feature_specification], test_dict["d"]
    )
    print(f"R2 Treatment (test): {r2_boost_d_test}")

    results_df_tab_tmp = pd.DataFrame(
        data=[
            [
                ols_outcome_train,
                ols_outcome_test,
                ols_treatment_train,
                ols_treatment_test,
            ],
            [r2_boost_q_train, r2_boost_q_test, r2_boost_d_train, r2_boost_d_test],
        ],
    )
    results_df_tab = pd.concat([results_df_tab, results_df_tab_tmp], axis=0)

row_names = [
    "OLS (Tabular)",
    "Boosting (Tabular)",
    "OLS (Tabular + PCA)",
    "Boosting (Tabular + PCA)",
    "OLS (Tabular + Similarities)",
    "Boosting (Tabular + Similarities)",
]

results_df_tab.columns = column_names
results_df_tab.index = row_names

results_df_tab


 ------------------------------------
Feature specification: x

OLS


R2 Outcome (train): 0.21370864863876404
R2 Outcome (test): 0.20838958759549508
R2 Treatment (train): 0.16152708611454358
R2 Treatment (test): 0.1581512127497423

Boosting


R2 Outcome (train): 0.760628763684153


R2 Outcome (test): 0.4750710061808484


R2 Treatment (train): 0.7369382567656562
R2 Treatment (test): 0.18535617147478323

 ------------------------------------
Feature specification: x_pca

OLS


...

R2 Outcome (train): 0.894076959357001


R2 Outcome (test): 0.5492763363033921


R2 Treatment (train): 0.9437429052002093


R2 Treatment (test): 0.3462135579490163


,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.213709,0.208390,0.161527,0.158151
Boosting (Tabular),0.760629,0.475071,0.736938,0.185356
OLS (Tabular + PCA),0.303647,0.300047,0.173801,0.167796
Boosting (Tabular + PCA),0.896155,0.558810,0.949570,0.393702
OLS (Tabular + Similarities),0.319832,0.312390,0.171768,0.168592
Boosting (Tabular + Similarities),0.894077,0.549276,0.943743,0.346214


Summary

In [10]:
results_df = pd.concat([results_df_tab, results_df_deep], axis=0)
results_df

,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.213709,0.208390,0.161527,0.158151
Boosting (Tabular),0.760629,0.475071,0.736938,0.185356
OLS (Tabular + PCA),0.303647,0.300047,0.173801,0.167796
Boosting (Tabular + PCA),0.896155,0.558810,0.949570,0.393702
OLS (Tabular + Similarities),0.319832,0.312390,0.171768,0.168592
Boosting (Tabular + Similarities),0.894077,0.549276,0.943743,0.346214
Deep Time Independent,0.853900,0.503100,0.953900,0.657400
Deep Time Dependent,0.861100,0.606500,0.958900,0.648900


## Performance Diff Data

In [11]:
feature_specifications = ["x", "x_pca", "x_sim"]
results_df_diff_tab = pd.DataFrame()

for feature_specification in feature_specifications:
    print("\n ------------------------------------")
    print(f"Feature specification: {feature_specification}")

    print("\nOLS")
    outcome_reg = sm.OLS(train_dict["y_diff"], train_dict[feature_specification]).fit()
    ols_outcome_train = r2_score(
        train_dict["y_diff"], outcome_reg.predict(train_dict[feature_specification])
    )
    print(f"R2 Outcome (train): {ols_outcome_train}")
    ols_outcome_test = r2_score(
        test_dict["y_diff"], outcome_reg.predict(test_dict[feature_specification])
    )
    print(f"R2 Outcome (test): {ols_outcome_test}")

    treatment_reg = sm.OLS(
        train_dict["d_diff"], train_dict[feature_specification]
    ).fit()
    ols_treatment_train = r2_score(
        train_dict["d_diff"], treatment_reg.predict(train_dict[feature_specification])
    )
    print(f"R2 Treatment (train): {ols_treatment_train}")
    ols_treatment_test = r2_score(
        test_dict["d_diff"], treatment_reg.predict(test_dict[feature_specification])
    )
    print(f"R2 Treatment (test): {ols_treatment_test}")

    print("\nBoosting")
    boost_q_outcome = LGBMRegressor(
        n_estimators=500, lr=0.02, random_state=42, verbose=-1
    )
    boost_q_outcome.fit(train_dict[feature_specification], train_dict["y_diff"])
    r2_boost_q_train = boost_q_outcome.score(
        train_dict[feature_specification], train_dict["y_diff"]
    )
    print(f"R2 Outcome (train): {r2_boost_q_train}")
    r2_boost_q_test = boost_q_outcome.score(
        test_dict[feature_specification], test_dict["y_diff"]
    )
    print(f"R2 Outcome (test): {r2_boost_q_test}")

    boost_q_treatment = LGBMRegressor(
        n_estimators=500, lr=0.02, random_state=42, verbose=-1
    )
    boost_q_treatment.fit(train_dict[feature_specification], train_dict["d_diff"])
    r2_boost_d_train = boost_q_treatment.score(
        train_dict[feature_specification], train_dict["d_diff"]
    )
    print(f"R2 Treatment (train): {r2_boost_d_train}")
    r2_boost_d_test = boost_q_treatment.score(
        test_dict[feature_specification], test_dict["d_diff"]
    )
    print(f"R2 Treatment (test): {r2_boost_d_test}")

    results_df_diff_tab_tmp = pd.DataFrame(
        data=[
            [
                ols_outcome_train,
                ols_outcome_test,
                ols_treatment_train,
                ols_treatment_test,
            ],
            [r2_boost_q_train, r2_boost_q_test, r2_boost_d_train, r2_boost_d_test],
        ],
    )
    results_df_diff_tab = pd.concat(
        [results_df_diff_tab, results_df_diff_tab_tmp], axis=0
    )

row_names = [
    "OLS (Tabular)",
    "Boosting (Tabular)",
    "OLS (Tabular + PCA)",
    "Boosting (Tabular + PCA)",
    "OLS (Tabular + Similarities)",
    "Boosting (Tabular + Similarities)",
]

results_df_diff_tab.columns = column_names
results_df_diff_tab.index = row_names

results_df_diff_tab


 ------------------------------------
Feature specification: x

OLS
R2 Outcome (train): 0.07640248387937332
R2 Outcome (test): 0.07077318277316202
R2 Treatment (train): 0.005791066894051733
R2 Treatment (test): 0.006365797810488738

Boosting


R2 Outcome (train): 0.3513182806246494


R2 Outcome (test): 0.1513066591847848


R2 Treatment (train): 0.23568023994092024
R2 Treatment (test): -0.004267409128240862

 ------------------------------------
Feature specification: x_pca

OLS


R2 Outcome (train): 0.07646317146803672
R2 Outcome (test): 0.0708775606661376
R2 Treatment (train): 0.006378811202313872
R2 Treatment (test): 0.006727487158112555

Boosting


...

R2 Treatment (train): 0.3184695346889267
R2 Treatment (test): 0.011242513221695893

 ------------------------------------
Feature specification: x_sim

OLS


R2 Outcome (train): 0.07646872119150894
R2 Outcome (test): 0.0708629532168974
R2 Treatment (train): 0.006302450184673791
R2 Treatment (test): 0.006642170110467749

Boosting


R2 Outcome (train): 0.4487700079379012
R2 Outcome (test): 0.18581661843023833


R2 Treatment (train): 0.3114318895482765
R2 Treatment (test): 0.007725296926115899


,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.076402,0.070773,0.005791,0.006366
Boosting (Tabular),0.351318,0.151307,0.235680,-0.004267
OLS (Tabular + PCA),0.076463,0.070878,0.006379,0.006727
Boosting (Tabular + PCA),0.453135,0.181254,0.318470,0.011243
OLS (Tabular + Similarities),0.076469,0.070863,0.006302,0.006642
Boosting (Tabular + Similarities),0.448770,0.185817,0.311432,0.007725


Summary

In [12]:
results_df_diff = pd.concat([results_df_diff_tab, results_df_diff_deep], axis=0)
results_df_diff

,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.076402,0.070773,0.005791,0.006366
Boosting (Tabular),0.351318,0.151307,0.235680,-0.004267
OLS (Tabular + PCA),0.076463,0.070878,0.006379,0.006727
Boosting (Tabular + PCA),0.453135,0.181254,0.318470,0.011243
OLS (Tabular + Similarities),0.076469,0.070863,0.006302,0.006642
Boosting (Tabular + Similarities),0.448770,0.185817,0.311432,0.007725
Deep Time Independent,0.194322,0.144769,0.010760,0.010119
Deep Time Dependent,0.102134,0.094789,0.012786,0.012397


## Summary

In [13]:
results_df[["R2 Q Test", "R2 P Test"]].round(4) * 100

,R2 Q Test,R2 P Test
OLS (Tabular),20.84,15.82
Boosting (Tabular),47.51,18.54
OLS (Tabular + PCA),30.00,16.78
Boosting (Tabular + PCA),55.88,39.37
OLS (Tabular + Similarities),31.24,16.86
Boosting (Tabular + Similarities),54.93,34.62
Deep Time Independent,50.31,65.74
Deep Time Dependent,60.65,64.89


In [14]:
results_df_diff[["R2 Q Test", "R2 P Test"]].round(4) * 100

,R2 Q Test,R2 P Test
OLS (Tabular),7.08,0.64
Boosting (Tabular),15.13,-0.43
OLS (Tabular + PCA),7.09,0.67
Boosting (Tabular + PCA),18.13,1.12
OLS (Tabular + Similarities),7.09,0.66
Boosting (Tabular + Similarities),18.58,0.77
Deep Time Independent,14.48,1.01
Deep Time Dependent,9.48,1.24
